In [1]:
import pandas as pd
import numpy as np

In [71]:
# Работа с текстом
# Векторизованные методы через str
# - lower, upper
# - strip, lstrip, rstrip
# - replace
# - contains
# - split
# - extract
# Данные с https://store.poidata.xyz/ru
# Задача:
# 1) Скачать датасет с store.poidata.xyz по России
# 2) Подсчитать кол-во адресов по регионам (столбец prov)
# - очиситить данные и привести к единообразию
# 3) Подсчитать домены верхнего уровня (ru, com, info ...)
# 4) Из formatted_address вытащить почтовый индекс (6 цифр)

In [63]:
df_raw = pd.read_csv('../data/raw/ru_sample.csv')

In [17]:
df_city = df_raw[['city', 'neighborhood']].copy()

In [18]:
df_city

,city,neighborhood
0,Москва,Moscow
1,Domodedovo,Домодедово
2,Падар,Весенняя
3,Краснодар,Krasnodar
4,Vladivostok,Владивосток
...,...,...
1365,Москва,NaN
1366,Черкесск,Cherkessk
1367,Makhachkala,NaN
1368,Москва,Moscow


In [19]:
bad_words = [
    'сельское поселение',
    'с.',
    'г.',
    'городской округ',
    'город',
    'район',
    'поселок',
]

In [20]:
df_city['clean_city'] = df_city['city'].str.lower()
df_city['clean_neighborhood'] = df_city['neighborhood'].str.lower()

In [21]:
for bad_word in bad_words:
    df_city['clean_city'] = df_city['clean_city'].str.replace(bad_word, '')
    df_city['clean_neighborhood'] = df_city['clean_neighborhood'].str.replace(bad_word, '')

In [22]:
df_city['clean_city'] = df_city['clean_city'].str.strip()
df_city['clean_neighborhood'] = df_city['clean_neighborhood'].str.strip()

In [23]:
df_city[df_city['city']=='г. Москва']

,city,neighborhood,clean_city,clean_neighborhood
366,г. Москва,Moscow,москва,moscow


In [31]:
df_city[df_city['clean_city'].str.contains('бург')]['clean_city'].value_counts()

clean_city
санкт-петербург    109
оренбург             8
Name: count, dtype: int64

In [34]:
pat_rus = r'[а-я]'

In [37]:
df_city['city_is_rus'] = df_city['clean_city'].str.contains(pat_rus)
df_city['neighborhood_is_rus'] = df_city['clean_neighborhood'].str.contains(pat_rus)

In [38]:
df_city

,city,neighborhood,clean_city,clean_neighborhood,city_is_rus,neighborhood_is_rus
0,Москва,Moscow,москва,moscow,True,False
1,Domodedovo,Домодедово,domodedovo,домодедово,False,True
2,Падар,Весенняя,падар,весенняя,True,True
3,Краснодар,Krasnodar,краснодар,krasnodar,True,False
4,Vladivostok,Владивосток,vladivostok,владивосток,False,True
...,...,...,...,...,...,...
1365,Москва,NaN,москва,NaN,True,NaN
1366,Черкесск,Cherkessk,черкесск,cherkessk,True,False
1367,Makhachkala,NaN,makhachkala,NaN,False,NaN
1368,Москва,Moscow,москва,moscow,True,False


In [44]:
def select_city_rus(row):
    if row['city_is_rus'] is True:
        return row['clean_city']
    elif row['neighborhood_is_rus'] is True:
        return row['clean_neighborhood']
    else:
        return np.nan

In [45]:
df_city['city_rus'] = df_city.apply(select_city_rus, axis=1)

In [47]:
df_city[['city', 'neighborhood', 'city_rus']]

,city,neighborhood,city_rus
0,Москва,Moscow,москва
1,Domodedovo,Домодедово,домодедово
2,Падар,Весенняя,падар
3,Краснодар,Krasnodar,краснодар
4,Vladivostok,Владивосток,владивосток
...,...,...,...
1365,Москва,NaN,москва
1366,Черкесск,Cherkessk,черкесск
1367,Makhachkala,NaN,NaN
1368,Москва,Moscow,москва


In [64]:
df_2 = df_raw[['email']].copy()

In [65]:
df_2

,email
0,info@biracs.ru
1,zakaz@mebel-domodedovo.ru
2,info@soleniyamagnat.ru
3,krasnodar@moskit.info
4,complect@vladivostok.ru
...,...
1365,info@kiwident.ru
1366,interoffice.rsue@gmail.com
1367,support.north-america@liqui-moly.com
1368,fizkult@teoriya.ru


In [66]:
df_2['email_name'] = df_2['email'].str.split('@', expand=True)[0]
df_2['email_domain'] = df_2['email'].str.split('@', expand=True)[1]

In [67]:
df_2['email_domain'].value_counts()

email_domain
mail.ru                  151
gmail.com                 68
yandex.ru                 64
bk.ru                     18
list.ru                   10
                        ... 
gsen.ru                    1
mfa.gov.by                 1
kiwident.ru                1
liqui-moly.com             1
zelenaya-energiya.run      1
Name: count, Length: 984, dtype: int64

In [68]:
df_3 = df_raw[['adr']].copy()

In [69]:
df_3['adr_num'] = df_3['adr'].str.extract(r'(\d+)')

In [70]:
df_3

,adr,adr_num
0,Leninskiy Prospekt 30,30
1,Kashirskoye Shosse 17а,17
2,Строительная улица 15,15
3,ul. Sormovskaya 3/7,3
4,Prospekt Krasnogo Znameni 34,34
...,...,...
1365,Высокая улица 4,4
1366,Krasnaya Ulitsa 3,3
1367,пос Семендер,NaN
1368,Tverskaya St 13,13
